In [1]:
!mkdir -p /kaggle/working/code
!mkdir -p /kaggle/working/CROMA

In [12]:
!cp /kaggle/input/datasets/anamikapatel8/croma-base/CROMA_base.pt /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/pretrain_croma.py /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/use_croma.py /kaggle/working/CROMA/


In [2]:
!pip install rasterio

In [ ]:
import os
import json

# Confirm these paths match your "Data" path 
S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"

def get_chip_id(folder_name):
    # Extracts the coordinate suffix, e.g., '00905-08320'
    return folder_name.split("_")[-1]

def build_file_list():
    if not os.path.exists(S1_ROOT) or not os.path.exists(S2_ROOT):
        print(f"ERROR: Root paths not found. Please verify S1_ROOT and S2_ROOT.")
        return []

    s1_folders = sorted(os.listdir(S1_ROOT))
    s2_folders = sorted(os.listdir(S2_ROOT))
    s2_dict = {get_chip_id(f): f for f in s2_folders}
    
    pairs = []
    
    # Required files for a "valid" research pair
    s1_required = ["VV.tif", "VH.tif", "LabelWater.tif"]
    # CROMA expects 12 bands (B1 through B11 or B12 depending on version)
    s2_required = [f"B{i}.tif" for i in range(1, 10)] + ["B8A.tif", "B11.tif", "B12.tif"]

    print("--- Starting File Integrity Audit ---")

    for s1_f in s1_folders:
        cid = get_chip_id(s1_f)
        if cid in s2_dict:
            s1_path = os.path.join(S1_ROOT, s1_f)
            s2_path = os.path.join(S2_ROOT, s2_dict[cid])
            
            # Verifying SAR files exist for this chip
            s1_valid = all(os.path.exists(os.path.join(s1_path, f)) for f in s1_required)
            
            # Verifying multi-spectral files exist for this chip
            s2_valid = all(os.path.exists(os.path.join(s2_path, f)) for f in s2_required)
            
            if s1_valid and s2_valid:
                pairs.append((s1_f, s2_dict[cid]))

    print(f"Total S1 folders: {len(s1_folders)}")
    print(f"Total S2 folders: {len(s2_folders)}")
    print(f"Successfully matched and verified pairs: {len(pairs)}")
    
    if len(pairs) > 0:
        print(f"\nExample Matched Pair:")
        print(f" S1: {pairs[100][0]}")
        print(f" S2: {pairs[100][1]}")
    
    return pairs

matched_pairs = build_file_list()

# Saves to a JSON file 
with open("/kaggle/working/matched_pairs.json", "w") as f:
    json.dump(matched_pairs, f)

print(f"Successfully saved {len(matched_pairs)} pairs to matched_pairs.json")

--- Starting File Integrity Audit ---
Total S1 folders: 900
Total S2 folders: 900
Successfully matched and verified pairs: 900

Example Matched Pair:
 S1: S1A_IW_GRDH_1SDV_20180507T160424_20180507T160449_021800_025A09_981B_01536-10752
 S2: 20180507T074611_20180507T080728_T36NXF_01536-10752
Successfully saved 900 pairs to matched_pairs.json


In [ ]:
%%writefile /kaggle/working/code/dataset_c2sms.py 
import sys
sys.path.append("/kaggle/working/code")
import os
import numpy as np
import torch
from torch.utils.data import Dataset
import rasterio
from scipy.ndimage import uniform_filter

def refined_lee_filter(img, size=7):
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    img_mean = uniform_filter(img, (size, size))
    img_sqr_mean = uniform_filter(img**2, (size, size))
    img_variance = np.maximum(0, img_sqr_mean - img_mean**2)
    overall_mean = np.mean(img)
    if overall_mean == 0: return img
    overall_variance = np.var(img)
    denominator = img_variance + (overall_variance / (overall_mean**2 + 1e-6))
    img_weights = np.clip(img_variance / (denominator + 1e-6), 0, 1)
    return img_mean + img_weights * (img - img_mean)

class C2SMSDataset(Dataset):
    def __init__(self, s1_root, s2_root, matched_pairs):
        self.s1_root = s1_root
        self.s2_root = s2_root
        self.pairs = matched_pairs
        self.eo_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

    def __len__(self):
        return len(self.pairs)

    def _preprocess_sar(self, sar):
        sar_db = 10.0 * np.log10(np.clip(sar, 1e-8, None))
        p1, p99 = np.percentile(sar_db, 1), np.percentile(sar_db, 99)
        sar_db = np.clip(sar_db, p1, p99)
        return ((sar_db - p1) / (p99 - p1 + 1e-6) * 2.0 - 1.0).astype(np.float32)

    def _preprocess_eo(self, eo):
        eo = np.nan_to_num(eo, nan=0.0)
        return (np.clip(eo, 0, 10000) / 10000.0 * 2.0 - 1.0).astype(np.float32)

    def __getitem__(self, idx):
        s1_folder, s2_folder = self.pairs[idx]
        s1_p, s2_p = os.path.join(self.s1_root, s1_folder), os.path.join(self.s2_root, s2_folder)
        
        # Loading SAR 
        with rasterio.open(os.path.join(s1_p, "VV.tif")) as src:
            vv = self._preprocess_sar(refined_lee_filter(src.read(1)))
        with rasterio.open(os.path.join(s1_p, "VH.tif")) as src:
            vh = self._preprocess_sar(refined_lee_filter(src.read(1)))
        sar_tensor = torch.from_numpy(np.stack([vv, vh], axis=0))

        # Loading flood labels and cloud mask
        with rasterio.open(os.path.join(s1_p, "LabelWater.tif")) as src:
            s1_label = torch.from_numpy((src.read(1) == 1).astype(np.int64))
        with rasterio.open(os.path.join(s2_p, "LabelWater.tif")) as src:
            s2_label = torch.from_numpy((src.read(1) == 1).astype(np.int64))
        with rasterio.open(os.path.join(s2_p, "LabelCloud.tif")) as src:
            cloud = torch.from_numpy((src.read(1) == 1).astype(np.float32))

        # Loading and preprocessing multi-spectral bands
        eo_data = [self._preprocess_eo(rasterio.open(os.path.join(s2_p, f"{b}.tif")).read(1)) for b in self.eo_bands]
        eo_tensor = torch.from_numpy(np.stack(eo_data, axis=0))

        return sar_tensor, eo_tensor, s1_label, s2_label, cloud

Writing /kaggle/working/code/dataset_c2sms.py


In [ ]:
%%writefile /kaggle/working/code/model.py
#CAGF
import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2 + 1, dim),
            nn.ReLU(),
            nn.Linear(dim, 1),
            nn.Sigmoid()
        )

    def forward(self, sar_feats, eo_feats, cloud_mask):
        B, C, H, W = sar_feats.shape
        sar_vec = F.adaptive_avg_pool2d(sar_feats, 1).view(B, C)
        eo_vec  = F.adaptive_avg_pool2d(eo_feats,  1).view(B, C)
        cloud_ratio = F.adaptive_avg_pool2d(cloud_mask.unsqueeze(1), 1).view(B, 1)
        combined = torch.cat([sar_vec, eo_vec, cloud_ratio], dim=1)
        g = self.gate(combined).view(B, 1, 1, 1)   # g≈1 means high cloud cover, trusting SAR; g≈0 means clear sky, trusting multi-spectral
        return g * sar_feats + (1 - g) * eo_feats

class CROMASegmentation(nn.Module):
    def __init__(self, croma_model, feat_dim, patch_size, mode="joint"):
        super().__init__()
        self.croma      = croma_model
        self.feat_dim   = feat_dim
        self.patch_size = patch_size
        self.mode       = mode

        if mode == "joint":
            self.fusion = GatedFusion(feat_dim)

        self.head = nn.Sequential(
            nn.Conv2d(feat_dim, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.Conv2d(256, 2, kernel_size=1)
        )

        for p in self.croma.parameters():
            p.requires_grad = False

    def forward(self, sar=None, eo=None, cloud=None):
        B, _, H, W = eo.shape if eo is not None else sar.shape
        h, w = H // self.patch_size, W // self.patch_size

        # ── CROMA backbone (frozen) ──────────────────────────────
        with torch.no_grad():
            out = self.croma(SAR_images=sar, optical_images=eo)

        # ── Branch on mode ───────────────────────────────────────────────
        if self.mode == "joint":
            # Reshape tokens -> 2-D feature maps  (gradients flow through here)
            s_feats = out["SAR_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)
            e_feats = out["optical_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

            cloud_small = F.interpolate(
                cloud.unsqueeze(1).float(), size=(h, w), mode='nearest'
            ).squeeze(1)

            feats = self.fusion(s_feats, e_feats, cloud_small)  # gate trains here

        elif self.mode == "eo":
            feats = out["optical_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

        else:  # "sar"
            feats = out["SAR_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

        # ── Upsample -> segmentation head ────────────────────────────────
        feats = F.interpolate(feats, size=(H, W), mode="bilinear", align_corners=False)
        return self.head(feats)

Writing /kaggle/working/code/model.py


In [ ]:
%%writefile /kaggle/working/code/model.py
#Patch-wise gated fusion 
import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(dim * 2 + 1, dim, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(dim, 1, kernel_size=1),
            nn.Sigmoid()
        )
        # New fusion layer: takes concatenated features + gate output
        # This allows for complex, non-linear interactions
        self.fusion_conv = nn.Sequential(
            nn.Conv2d(dim * 2 + 1, dim, kernel_size=1), # Input: (SAR_feats + EO_feats + g) -> dim
            nn.BatchNorm2d(dim), # Adds batch norm for stability
            nn.ReLU() # Adds activation
        )

    def forward(self, sar_feats, eo_feats, cloud_mask):
        # Calculates the patch-wise gate 'g' based on all inputs
        combined_for_gate = torch.cat([sar_feats, eo_feats, cloud_mask.unsqueeze(1)], dim=1)
        g = self.gate(combined_for_gate)   # g will be (B, 1, h, w)

        # Concatenates sar_feats, eo_feats, and the learned gate 'g'
        # The fusion_conv layer will learn how to combine these.
        fused_input = torch.cat([sar_feats, eo_feats, g], dim=1)
        feats = self.fusion_conv(fused_input) # Applying convolution for complex interaction

        return feats # Returns the result of the new fusion layer

class CROMASegmentation(nn.Module):
    def __init__(self, croma_model, feat_dim, patch_size, mode="joint"):
        super().__init__()
        self.croma      = croma_model
        self.feat_dim   = feat_dim
        self.patch_size = patch_size
        self.mode       = mode

        if mode == "joint":
            self.fusion = GatedFusion(feat_dim)

        self.head = nn.Sequential(
            nn.Conv2d(feat_dim, 256, kernel_size=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.Conv2d(256, 2, kernel_size=1)
        )

        for p in self.croma.parameters():
            p.requires_grad = False

    def forward(self, sar=None, eo=None, cloud=None):
        B, _, H, W = eo.shape if eo is not None else sar.shape
        h, w = H // self.patch_size, W // self.patch_size

        # ── CROMA backbone (frozen) ──────────────────────────────
        with torch.no_grad():
            out = self.croma(SAR_images=sar, optical_images=eo)

        # ── Branch on mode ───────────────────────────────────────────────
        if self.mode == "joint":
            # Reshape tokens -> 2-D feature maps  (gradients flow through here)
            s_feats = out["SAR_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)
            e_feats = out["optical_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

            cloud_small = F.interpolate(
                cloud.unsqueeze(1).float(), size=(h, w), mode='nearest'
            ).squeeze(1)

            feats = self.fusion(s_feats, e_feats, cloud_small)  # gate trains here

        elif self.mode == "eo":
            feats = out["optical_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

        else:  # "sar"
            feats = out["SAR_encodings"].transpose(1, 2).reshape(B, self.feat_dim, h, w)

        # ── Upsample -> segmentation head ────────────────────────────────
        feats = F.interpolate(feats, size=(H, W), mode="bilinear", align_corners=False)
        return self.head(feats)

In [11]:
%%writefile /kaggle/working/code/croma_wrapper.py 
import sys
sys.path.append("/kaggle/working")
import torch
from CROMA.use_croma import PretrainedCROMA

def load_croma_encoder(pretrained_path, image_resolution, mode="joint", device="cpu"):
    """
    Loads CROMA backbone based on requested modality.
    mode: 'joint' (both), 'eo' (optical), or 'sar' (SAR)
    """
    modality_map = {"joint": "both", "eo": "optical", "sar": "SAR"}
    
    model = PretrainedCROMA(
        pretrained_path=pretrained_path,
        size="base",
        modality=modality_map[mode],
        image_resolution=image_resolution
    ).to(device)
    
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
        
    return model, model.encoder_dim, model.patch_size

Overwriting /kaggle/working/code/croma_wrapper.py


In [ ]:
%%writefile /kaggle/working/code/train_UP.py
import sys
sys.path.append("/kaggle/working/code")
import os, json, random
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm 
from torch.utils.data import DataLoader, Subset
from dataset_c2sms import C2SMSDataset
from model import CROMASegmentation
from croma_wrapper import load_croma_encoder 

# --- Reproducibility Setup ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROMA_CKPT = "/kaggle/working/CROMA/CROMA_base.pt"
S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
PAIRS_JSON = "/kaggle/working/matched_pairs.json"

RESUME_PATH = "/kaggle/input/datasets/anamikapatel8/croma-sar-eo-fused-checkpoint/croma_fused_checkpoint_UP.pth"

SAVE_PATH_MODEL = "/kaggle/working/croma_c2sms_fused_UP.pt"
SAVE_PATH_CHECKPOINT = "/kaggle/working/croma_fused_checkpoint_UP.pth"

with open(PAIRS_JSON, "r") as f:
    matched_pairs = json.load(f)

# --- Cloud-Aware Loss Function ---
def cloud_aware_loss(logits, s1_label, s2_label, cloud_mask, criterion):
    # Dynamic label selection: using optical for clear pixels, SAR for cloudy ones S2 for clear pixels, S1 for cloudy ones
    target = (s2_label * (1 - cloud_mask.long())) + (s1_label * cloud_mask.long())
    ce = criterion(logits, target)
    
    probs = torch.softmax(logits, dim=1)[:, 1]
    target_f = target.float()
    inter = (probs * target_f).sum(dim=(1, 2))        
    union = probs.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))  
    dice  = (1 - (2 * inter + 1e-6) / (union + 1e-6)).mean()  
        
    # Weighted CE+Dice ratio tuned to prevent class collapse
    return 0.2 * ce + 0.8 * dice

# --- Data Preparation ---
dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
indices = torch.randperm(len(dataset)).tolist()
split = int(0.8 * len(dataset))
train_idx, val_idx = indices[:split], indices[split:]

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

# --- Model Initialization (JOINT MODE) ---
print(">>> Initializing Fused (Joint) SAR+EO Model...")
# mode="joint" activates the Gated Fusion backbone
croma, f_dim, p_size = load_croma_encoder(CROMA_CKPT, 512, mode="joint", device=DEVICE)
model = CROMASegmentation(croma, f_dim, p_size, mode="joint").to(DEVICE)

# --- Optimizer Setup ---

optimizer = torch.optim.Adam([
    {'params': model.fusion.parameters(), 'lr': 1e-4},
    {'params': model.head.parameters(), 'lr': 5e-5}
])
# 15.0 weight to water class to combat extreme imbalance
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 15.0]).to(DEVICE))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# --- Training Loop ---
start_epoch = 0
best_iou = 0.0

if os.path.exists(RESUME_PATH):
    print(f">>> Resuming from checkpoint: {RESUME_PATH}")
    checkpoint = torch.load(RESUME_PATH, map_location=DEVICE)
    
    # Restores model and optimizer states
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    # Restores epoch and best metric
    start_epoch = checkpoint['epoch'] + 1
    best_iou = checkpoint.get('best_iou', 0.0)
    print(f">>> Resuming from Epoch {start_epoch} with Best IoU: {best_iou:.4f}")
else:
    print(">>> No checkpoint found at Resume Path. Starting from scratch.")
    
for epoch in range(start_epoch, 30):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [FUSED | Gated Fusion]")
    
    for sar, eo, s1_l, s2_l, cloud in pbar:
        sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE, non_blocking=True) for x in [sar, eo, s1_l, s2_l, cloud]]
        
        optimizer.zero_grad()
        # Joint forward pass: utilizes both sensors and the cloud mask
        logits = model(sar=sar, eo=eo, cloud=cloud) 
        
        loss = cloud_aware_loss(logits, s1_l, s2_l, cloud, criterion)
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # --- Validation ---
    model.eval()
    val_ious, val_dices = [], []
    with torch.no_grad():
        for sar, eo, s1_l, s2_l, cloud in val_loader:
            sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE, non_blocking=True) for x in [sar, eo, s1_l, s2_l, cloud]]
            
            logits = model(sar=sar, eo=eo, cloud=cloud)
            # Threshold=0.4: Optimized for fused confidence
            preds = (torch.softmax(logits, dim=1)[:, 1] > 0.4).long()
            target = (s2_l * (1 - cloud.long())) + (s1_l * cloud.long())
            
            intersection = (preds & target).sum().float()
            union = (preds | target).sum().float()
            val_ious.append((intersection + 1e-6) / (union + 1e-6))
            val_dices.append((2 * intersection + 1e-6) / (preds.sum() + target.sum() + 1e-6))
    
    avg_iou = torch.mean(torch.stack(val_ious)).item()
    avg_dice = torch.mean(torch.stack(val_dices)).item()
    
    print(f"Epoch {epoch:02d} | LR: {current_lr:.6f} | Loss: {epoch_loss/len(train_loader):.4f} | Val IoU: {avg_iou:.4f} | Val Dice: {avg_dice:.4f}")
    
    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), SAVE_PATH_MODEL)
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_iou': best_iou,
        }
        torch.save(checkpoint, SAVE_PATH_CHECKPOINT)
        print(f">>> Saved Best Fused Model and Checkpoint (IoU: {best_iou:.4f})")

Overwriting /kaggle/working/code/train_UP.py


In [ ]:
!python /kaggle/working/code/train_UP.py

>>> Initializing Fused (Joint) SAR+EO Model...
Initializing SAR encoder
Initializing optical encoder
Initializing joint SAR-optical encoder
>>> Resuming from checkpoint: /kaggle/input/datasets/anamikapatel8/croma-sar-eo-fused-checkpoint/croma_fused_checkpoint_UP.pth
>>> Resuming from Epoch 8 with Best IoU: 0.5727
Epoch 8 [FUSED | Gated Fusion]: 100%|█| 180/180 [28:07<00:00,  9.38s/it, loss=0.
Epoch 08 | LR: 0.000040 | Loss: 0.3938 | Val IoU: 0.5765 | Val Dice: 0.7060
>>> Saved Best Fused Model and Checkpoint (IoU: 0.5765)
Epoch 9 [FUSED | Gated Fusion]: 100%|█| 180/180 [28:23<00:00,  9.46s/it, loss=0.
Epoch 09 | LR: 0.000038 | Loss: 0.4001 | Val IoU: 0.5306 | Val Dice: 0.6616
Epoch 10 [FUSED | Gated Fusion]: 100%|█| 180/180 [28:21<00:00,  9.45s/it, loss=0
Epoch 10 | LR: 0.000035 | Loss: 0.3887 | Val IoU: 0.5775 | Val Dice: 0.7041
>>> Saved Best Fused Model and Checkpoint (IoU: 0.5775)
Epoch 11 [FUSED | Gated Fusion]: 100%|█| 180/180 [28:22<00:00,  9.46s/it, loss=0
Epoch 11 | LR: 0.0000

In [ ]:
%%writefile /kaggle/working/code/evaluate_comparison_all3.py
import sys
sys.path.append("/kaggle/working/code")
import os, json, torch, time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from dataset_c2sms import C2SMSDataset
from model import CROMASegmentation
from croma_wrapper import load_croma_encoder
from scipy.ndimage import uniform_filter

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROMA_CKPT = "/kaggle/working/CROMA/CROMA_base.pt"

# WEIGHT PATHS
EO_WEIGHTS = "/kaggle/input/datasets/anamikapatel8/croma-c2ms-eo-only/croma_c2sms_eo_only.pt"
SAR_WEIGHTS = "/kaggle/input/datasets/anamikapatel8/croma-sar-only/croma_c2sms_sar_only_best (3).pt"
GATED_WEIGHTS = "/kaggle/input/datasets/anamikapatel8/croma-c2sms-eo-sar-1/croma_c2sms_EO_SAR.pt"

S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
PAIRS_JSON = "/kaggle/working/matched_pairs.json"
RESULTS_DIR = "/kaggle/working/comparison_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Visibility Helpers ---
def refined_lee_filter(img, size=7):
    img_mean = uniform_filter(img, (size, size))
    img_sqr_mean = uniform_filter(img**2, (size, size))
    img_var = np.maximum(0, img_sqr_mean - img_mean**2)
    overall_var = np.var(img)
    weights = img_var / (img_var + overall_var + 1e-6)
    return img_mean + weights * (img - img_mean)

def enhance_eo(eo_tensor):
    eo = eo_tensor.cpu().numpy()
    eo = (eo * 0.5 + 0.5)
    rgb = np.stack([eo[3], eo[2], eo[1]], axis=-1)
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], 2), np.percentile(rgb[..., c], 98)
        rgb[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-6), 0, 1)
    return rgb ** 0.8

def enhance_sar(sar_tensor):
    vv = sar_tensor[0].cpu().numpy()
    vv = vv * 0.5 + 0.5
    vv = refined_lee_filter(vv, size=7)
    lo, hi = np.percentile(vv, 1), np.percentile(vv, 99)
    vv = np.clip((vv - lo) / (hi - lo + 1e-6), 0, 1)
    return vv ** 0.7

def get_metrics(preds, target):
    preds, target = preds.bool(), target.bool()
    tp = (preds & target).sum().float()
    fp = (preds & ~target).sum().float()
    fn = (~preds & target).sum().float()
    union = (preds | target).sum().float()
    iou = (tp + 1e-6) / (union + 1e-6)
    dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
    precision = (tp + 1e-6) / (tp + fp + 1e-6)
    recall = (tp + 1e-6) / (tp + fn + 1e-6)
    return [iou.item(), dice.item(), precision.item(), recall.item()]

# --- Model Initialization ---
with open(PAIRS_JSON, "r") as f:
    matched_pairs = json.load(f)

dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
torch.manual_seed(42)
indices = torch.randperm(len(dataset)).tolist()
split = int(0.8 * len(dataset))
val_idx = indices[split:]
val_loader = DataLoader(Subset(dataset, val_idx), batch_size=1, shuffle=False)

print(">>> Initializing Models (MS, SAR, and Gated Fusion)...")
# 1. MS-Only
croma_eo, f_dim, p_size = load_croma_encoder(CROMA_CKPT, 512, mode="eo", device=DEVICE)
model_eo = CROMASegmentation(croma_eo, f_dim, p_size, mode="eo").to(DEVICE)
model_eo.load_state_dict(torch.load(EO_WEIGHTS, map_location=DEVICE))

# 2. SAR-Only 
croma_sar, _, _ = load_croma_encoder(CROMA_CKPT, 512, mode="sar", device=DEVICE)
model_sar = CROMASegmentation(croma_sar, f_dim, p_size, mode="sar").to(DEVICE)
model_sar.load_state_dict(torch.load(SAR_WEIGHTS, map_location=DEVICE))

# 3. Gated Fusion
croma_joint, _, _ = load_croma_encoder(CROMA_CKPT, 512, mode="joint", device=DEVICE)
model_gated = CROMASegmentation(croma_joint, f_dim, p_size, mode="joint").to(DEVICE)
model_gated.load_state_dict(torch.load(GATED_WEIGHTS, map_location=DEVICE))

model_eo.eval(); model_sar.eval(); model_gated.eval()

results = {"eo": [], "sar": [], "gated": []}
count = 0
print(f"\n>>> Evaluating Cloudy Images (Triple Comparison)...")

with torch.no_grad():
    for sar, eo, s1_l, s2_l, cloud in tqdm(val_loader):
        cloud_ratio = cloud.mean().item()
        if cloud_ratio < 0.1: continue # Focus on cloudy samples
            
        sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE) for x in [sar, eo, s1_l, s2_l, cloud]]
        target = (s2_l * (1 - cloud.long())) + (s1_l * cloud.long())
        
        # Predicts EO-Only
        preds_eo = (torch.softmax(model_eo(eo=eo), dim=1)[:, 1] > 0.5).long()
        results["eo"].append(get_metrics(preds_eo, target))

        # Predicts SAR-Only 
        preds_sar = (torch.softmax(model_sar(sar=sar), dim=1)[:, 1] > 0.4).long() # Lower thr for SAR
        results["sar"].append(get_metrics(preds_sar, target))
        
        # Predicts Gated Fusion
        preds_gated = (torch.softmax(model_gated(sar=sar, eo=eo, cloud=cloud), dim=1)[:, 1] > 0.4).long()
        m_gated = get_metrics(preds_gated, target)
        results["gated"].append(m_gated)

        if m_gated[0] > 0.50 and count < 50:
            fig, ax = plt.subplots(2, 4, figsize=(24, 12)) # Expanded to 4 columns
        
            rgb = enhance_eo(eo[0])
            sar_vis = enhance_sar(sar[0])
            target_np = target[0].cpu().numpy()
            cloud_np = cloud[0].cpu().numpy()
        
            # Top Row: Inputs
            ax[0, 0].imshow(rgb); ax[0, 0].set_title(f"EO RGB (Cloud: {cloud_ratio:.2f})"); ax[0, 0].axis("off")
            ax[0, 1].imshow(sar_vis, cmap="gray"); ax[0, 1].set_title("SAR VV"); ax[0, 1].axis("off")
            ax[0, 2].imshow(cloud_np, cmap="gray"); ax[0, 2].set_title("Cloud Mask"); ax[0, 2].axis("off")
            ax[0, 3].imshow(target_np, cmap="Blues"); ax[0, 3].set_title("Hybrid Ground Truth"); ax[0, 3].axis("off")
        
            # Bottom Row: Preds
            # 1. MS Result 
            ax[1, 0].imshow(rgb); ax[1, 0].imshow(preds_eo[0].cpu().numpy(), cmap="Reds", alpha=0.5)
            ax[1, 0].set_title(f"EO-Only (IoU: {results['eo'][-1][0]:.3f})"); ax[1, 0].axis("off")

            # 2. SAR Result 
            ax[1, 1].imshow(sar_vis, cmap="gray"); ax[1, 1].imshow(preds_sar[0].cpu().numpy(), cmap="Oranges", alpha=0.5)
            ax[1, 1].set_title(f"SAR-Only (IoU: {results['sar'][-1][0]:.3f})"); ax[1, 1].axis("off")

            # 3. Gated Fusion Result 
            ax[1, 2].imshow(rgb); ax[1, 2].imshow(preds_gated[0].cpu().numpy(), cmap="Greens", alpha=0.5)
            ax[1, 2].set_title(f"Gated Fusion (IoU: {results['gated'][-1][0]:.3f})"); ax[1, 2].axis("off")

            # 4. Error Map (Fusion vs GT)
            error_map = (preds_gated[0].cpu().numpy() != target_np)
            ax[1, 3].imshow(error_map, cmap="magma"); ax[1, 3].set_title("Fusion Error Map"); ax[1, 3].axis("off")
        
            plt.tight_layout()
            plt.savefig(f"{RESULTS_DIR}/comparison_sample_{count}.png", dpi=200)
            plt.close()
            count += 1

# Statistics and Summary
eo_m, sar_m, gated_m = np.mean(results["eo"], 0), np.mean(results["sar"], 0), np.mean(results["gated"], 0)
metrics = ["IoU", "Dice/F1", "Precision", "Recall"]
print("\n" + "="*70)
print(f"{'Metric':<12} | {'EO-Only':<10} | {'SAR-Only':<10} | {'Gated Fusion':<12} | {'Gain':<10}")
print("-" * 70)
for i, m in enumerate(metrics):
    imp = ((gated_m[i] - eo_m[i]) / eo_m[i]) * 100
    print(f"{m:<12} | {eo_m[i]:.4f}  | {sar_m[i]:.4f}  | {gated_m[i]:.4f}     | {imp:+.2f}%")
print("="*70)

In [ ]:
%%writefile /kaggle/working/code/find_optimal_weights.py
import sys
sys.path.append("/kaggle/working/code")
import os, json, torch, random
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset_c2sms import C2SMSDataset
from model import CROMASegmentation
from croma_wrapper import load_croma_encoder

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROMA_CKPT = "/kaggle/working/CROMA/CROMA_base.pt"
S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
PAIRS_JSON = "/kaggle/working/matched_pairs.json"

# --- 1. Calculates Ratio using ALL samples ---
def get_dataset_class_ratio(dataset):
    print(f"Calculating class ratio from ALL {len(dataset)} samples...")
    water_pixels = 0
    total_pixels = 0
    
    # We loop through every single image in the dataset
    for i in range(len(dataset)):
        _, _, s1_l, _, _ = dataset[i]
        water_pixels += torch.sum(s1_l == 1).item()
        total_pixels += s1_l.numel()
    
    land_pixels = total_pixels - water_pixels
    ratio = land_pixels / (water_pixels + 1e-6)
    return ratio

# --- 2. Metric for Grid Search ---
def evaluate_weight_combo(model, loader, class_weight):
    model.eval()
    ious = []
    with torch.no_grad():
        for sar, eo, s1_l, s2_l, cloud in loader:
            sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE) for x in [sar, eo, s1_l, s2_l, cloud]]
            target = (s2_l * (1 - cloud.long())) + (s1_l * cloud.long())
            
            logits = model(sar=sar, eo=eo, cloud=cloud)
            
            # --- Applying the class_weight to the water channel (index 1) ---
            # This shifts the model's bias towards water based on the search parameter
            logits[:, 1, :, :] *= class_weight 
            
            preds = (torch.softmax(logits, dim=1)[:, 1] > 0.5).long()
            
            intersection = (preds & target).sum().float()
            union = (preds | target).sum().float()
            ious.append((intersection + 1e-6) / (union + 1e-6))
            
    return torch.mean(torch.stack(ious)).item()

# --- Main Grid Search ---
with open(PAIRS_JSON, "r") as f:
    matched_pairs = json.load(f)

dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
ratio = get_dataset_class_ratio(dataset)
print(f"Detected Land-to-Water Pixel Ratio: {ratio:.2f}")

# Search Space 
class_weights = [0.3, 0.5, 0.7, 1.0] 
ce_dice_ratios = [(0.5, 0.5)]

# Loads Model once
croma, f_dim, p_size = load_croma_encoder(CROMA_CKPT, 512, mode="joint", device=DEVICE)
model = CROMASegmentation(croma, f_dim, p_size, mode="joint").to(DEVICE)
model.load_state_dict(torch.load("/kaggle/input/datasets/anamikapatel8/croma-c2sms-eo-sar/croma_c2sms_gated_best_1.pt", map_location=DEVICE))

# Full validation subset (Last 20% of the dataset)
indices = list(range(len(dataset)))
split = int(0.8 * len(dataset))
val_loader = DataLoader(Subset(dataset, indices[split:]), batch_size=4)

print(f"\n--- Starting Grid Search on {len(indices)-split} validation images ---")
best_score = 0
best_params = {}

for cw in class_weights:
    for ce_w, dice_w in ce_dice_ratios:
        # We evaluate the current model to see how it performs with these weights
        score = evaluate_weight_combo(model, val_loader, cw)
        print(f"Testing ClassWeight: {cw} | CE/Dice Ratio: {ce_w}/{dice_w} | Mean IoU: {score:.4f}")
        
        if score > best_score:
            best_score = score
            best_params = {"class_weight": cw, "ce_weight": ce_w, "dice_weight": dice_w}

print("\n" + "="*30)
print("OPTIMAL PARAMETERS FOUND:")
print(f"Class Weight: {best_params['class_weight']}")
print(f"CE Weight:    {best_params['ce_weight']}")
print(f"Dice Weight:  {best_params['dice_weight']}")
print("="*30)